In [ ]:
#!/usr/bin/env python
"""
LLM Tutoring Generation Script

This is the core generation script for the paper: it produces every one of
the 43,356-per-model tutor utterances analyzed in Results.

For each dialogue segment, we generate under:
  - 3 prompting strategies: baseline, few-shot, topic-context
  - 2 conditions per strategy: event (multimodal signal included) vs.
    no_event (text-only)
  - 2 independent runs per segment x condition combination (used later to
    check run-to-run consistency; see the MAI methodology)

That's 3 x 2 x 2 = 12 generations per segment, per model.

Usage:
    # Run all 4 models (default)
    python 03_generate_llm_responses.py

    # Run only one model
    python 03_generate_llm_responses.py --models "mistral:latest"

    # Custom input file
    python 03_generate_llm_responses.py --input mydata.csv

Input:  session_level_dialog_segments_final.csv (from 01_build_segments.py)
        math_topics.csv (from 02_extract_topics.py, reshaped so each row is
        transcriptID -> a single math_topics string for that session)

Output: llm_<model_tag>.csv, one per model, one row per generation.
        Supports resuming: if an output file already has rows in it,
        generation picks up right after the last completed segment
        rather than starting over.
"""

import argparse
import os
import re
from datetime import datetime
from typing import Optional

import ollama
import pandas as pd
from tqdm import tqdm

# ── Few-shot examples ---------------------------------------------------------
# Eleven hand-picked, REAL tutor lines from this same dataset (not
# independently authored), covering a spread of effective tutoring move
# types. Used only by the "fewshot" prompting strategy. These specific
# eleven segments were confirmed to NOT overlap with the evaluation set
# (see paper Methods, Section 3.2), so there's no data leakage.
FEWSHOT_EXAMPLES = (
    "## Examples of Effective Tutor Responses\n"
    "The following are real examples of how a good tutor responds during a session. "
    "They show a range of move types: Socratic questions, gentle corrections, encouragement, "
    "redirects, scaffolding, and social rapport. "
    "Study the style, length, and type of move before generating your response:\n\n"

    "-- Socratic questioning (prompting the student to reason, not just answer) --\n"
    "Example 1:\n"
    "Tutor: But you know what n equals, so why do I have an unknown variable when I know what the variable equals?\n\n"
    "Example 2:\n"
    "Tutor: And where would that line meet on the Y axis?\n\n"
    "Example 3:\n"
    "Tutor: If our rise is two and our run is three, what does that give us?\n\n"

    "-- Gentle correction (acknowledging partial correctness, redirecting the error) --\n"
    "Example 4:\n"
    "Tutor: Yeah, it's almost on two thousand and five, but not quite.\n\n"
    "Example 5:\n"
    "Tutor: Not quite, because you're adding them -- they're not the same quantity.\n\n"
    "Example 6:\n"
    "Tutor: Instead of multiplying, we do the opposite.\n\n"

    "-- Encouragement without giving away the answer --\n"
    "Example 7:\n"
    "Tutor: Keep going, you're doing well -- I'm not even here for the answer.\n\n"
    "Example 8:\n"
    "Tutor: You don't make mistakes -- don't doubt yourself.\n\n"

    "-- Scaffolding (breaking the problem into smaller steps) --\n"
    "Example 9:\n"
    "Tutor: Let's do the same thing for four times three, just like we did earlier.\n\n"
    "Example 10:\n"
    "Tutor: So you can go ahead and show that step -- subtract eight M over there.\n\n"

    "-- Redirect / focus (bringing student back on task) --\n"
    "Example 11:\n"
    "Tutor: Let's go ahead and do the closing, okay?\n\n"
)


def parse_args():
    """Command-line options -- lets you re-run just one model or point at a different input file."""
    parser = argparse.ArgumentParser(description="Generate LLM tutor responses.")
    parser.add_argument("--models", type=str,
                         default="wizardlm2:7b, mistral:latest, qwen3-vl:8b, gemma3:27b",
                         help='Comma-separated model names, e.g. "mistral:latest"')
    parser.add_argument("--input", type=str,
                         default="session_level_dialog_segments_final.csv",
                         help="Path to segments CSV (from 01_build_segments.py).")
    parser.add_argument("--topics", type=str,
                         default="math_topics.csv",
                         help="Path to per-session math topics CSV.")
    return parser.parse_args()


class TutorGenerationAgent:
    """
    Wraps a single Ollama model. Handles both prompting styles (baseline
    and few-shot use slightly different system prompts) and always runs
    the generated text through clean_generation() before returning it.
    """

    def __init__(self, model: str = "wizardlm2:7b", temperature: float = 0.2,
                 top_k: int = 5, provider: Optional[str] = None):
        self.model = model
        self.options = {"temperature": temperature, "top_k": top_k}
        self.provider = "ollama"

    def _system_prompt(self, fewshot: bool = False) -> str:
        """
        Two near-identical system prompts. The fewshot version adds a
        sentence acknowledging that examples will be shown; otherwise
        the instructions (guide without giving away the answer, output
        exactly one utterance, no labels) are the same.
        """
        if fewshot:
            return (
                "You are a math tutor generating ONLY the next tutor utterance in a tutoring dialogue. "
                "The student may think aloud and make mistakes. Your role is to guide them with questions, "
                "encouragement, and hints -- without giving away the solution. "
                "You will be shown examples of effective tutor responses, followed by a real dialogue. "
                "Please take into account the content and style of the previous contributions to the "
                "conversation when generating your next move -- match the level, tone, and pacing of the dialogue. "
                "Output ONLY the tutor's next single utterance -- no labels, no student lines, no explanations. "
                "Do NOT write 'Tutor:', do NOT continue the dialogue beyond one turn, do NOT simulate the student."
            )
        return (
            "You are a patient math tutor helping a novice student who is learning by problem solving. "
            "The student may think aloud and make mistakes. Your role is to guide them with questions, "
            "encouragement, and hints -- without giving away the solution. "
            "Please take into account the content and style of the previous contributions to the "
            "conversation when generating your next move -- match the level, tone, and pacing of the dialogue. "
            "Output ONLY your next single utterance as the tutor. "
            "Do NOT write 'Tutor:', do NOT generate student lines, do NOT continue past one turn."
        )

    def chat(self, content: str, fewshot: bool = False):
        """
        Send one generation request. Returns (cleaned_text, messages_sent,
        options_used, model_name) -- the extra return values are mostly
        useful for debugging/logging, only cleaned_text is used downstream.
        """
        messages = [
            {"role": "system", "content": self._system_prompt(fewshot=fewshot)},
            {"role": "user", "content": content},
        ]
        response = ollama.chat(model=self.model, messages=messages, options=self.options)
        raw = response["message"]["content"]
        return clean_generation(raw), messages, self.options, self.model


def clean_generation(text: str) -> str:
    """
    Post-processing safety net: even with explicit instructions, models
    sometimes still (a) prefix their answer with "Tutor:" anyway, or
    (b) hallucinate a fake student response after their own line. This
    strips both, so what we save is ALWAYS just the tutor's own words.
    """
    # Remove a leading "Tutor:" / "Assistant:" / "Teacher:" label, if present.
    text = re.sub(r"^(tutor|assistant|teacher)\s*:\s*", "", text.strip(), flags=re.IGNORECASE)

    # If the model kept going and wrote a fake student turn, cut everything
    # from that point onward -- we only want the tutor's single utterance.
    student_pattern = re.compile(r"\n\s*(student|s\s*:|learner)\s*:", re.IGNORECASE)
    match = student_pattern.search(text)
    if match:
        text = text[:match.start()].strip()

    # Belt-and-suspenders: also strip any remaining inline "Student:" text
    # that wasn't on its own line (regex above only catches newline-prefixed cases).
    text = re.sub(r"(student|learner)\s*:.*", "", text, flags=re.IGNORECASE)
    return text.strip()


def _output_rules() -> str:
    """
    Shared closing instructions appended to EVERY prompt, regardless of
    strategy -- keeps output format enforcement consistent across
    baseline / few-shot / topic-context so we're not accidentally
    comparing apples to oranges due to inconsistent instructions.
    """
    return (
        "Generate ONLY the tutor's next single utterance. "
        "Rules: (1) Output one sentence or question only. "
        "(2) Do NOT write 'Tutor:' or any label. "
        "(3) Do NOT generate any student lines. "
        "(4) Do NOT explain your reasoning. "
        "Output nothing except the tutor utterance itself."
    )


def make_prompt(dialog: str, event: Optional[str] = None) -> str:
    """
    BASELINE prompt. If `event` is provided (the event condition), it's
    inserted as a labeled "Contextual Event" section BEFORE the dialogue
    history -- this is the ONLY difference between the event and no_event
    conditions across all three strategies.
    """
    if pd.notna(event) and str(event).strip() != "":
        return (
            "## Contextual Event\n"
            "Something that occurred during this tutoring session (e.g. a student action, emotional signal, "
            "or pause). Use this to inform your tone and response where relevant, but do not let it dominate:\n"
            f"{event}\n\n"
            "## Dialogue So Far\n"
            f"{dialog}\n\n"
            f"{_output_rules()}"
        )
    return f"## Dialogue So Far\n{dialog}\n\n{_output_rules()}"


def make_prompt_fewshot(dialog: str, event: Optional[str] = None) -> str:
    """
    FEW-SHOT prompt: identical to make_prompt(), just with FEWSHOT_EXAMPLES
    prepended so the model sees eleven real tutor moves before the actual
    dialogue it needs to continue.
    """
    if pd.notna(event) and str(event).strip() != "":
        return (
            FEWSHOT_EXAMPLES +
            "## Contextual Event\n"
            "Something that occurred during this tutoring session (e.g. a student action, emotional signal, "
            "or pause). Use this to inform your tone and response where relevant, but do not let it dominate:\n"
            f"{event}\n\n"
            "## Dialogue So Far\n"
            f"{dialog}\n\n"
            f"{_output_rules()}"
        )
    return FEWSHOT_EXAMPLES + f"## Dialogue So Far\n{dialog}\n\n{_output_rules()}"


def make_prompt_topic_context(dialog: str, math_topics: str, event: Optional[str] = None) -> str:
    """
    TOPIC-CONTEXT prompt: prepends the session's extracted math topics
    (from 02_extract_topics.py) to the baseline prompt, so the model knows
    what subject matter this session covers without being told to
    introduce anything the student hasn't already raised.
    """
    return (
        "The following math concepts are being covered in this tutoring session. Use these to understand "
        "the subject matter and frame your response appropriately, but do not introduce new topics the "
        "student hasn't raised:\n"
        f"{math_topics}\n\n"
        f"{make_prompt(dialog, event=event)}"
    )


def append_row(rows: list, obs_id: int, model_name: str, timestamp: str,
                cond: str, exp: str, prompt: str, generation: str, options: dict, row: pd.Series):
    """
    Package up everything about one single generation into a dict, ready
    to be turned into a CSV row. Keeping this in one function avoids
    repeating this same 13-field dict six separate times below.
    """
    rows.append({
        "observation_id": obs_id,
        "LLM_version": model_name,
        "timestamp": timestamp,
        "context_condition": cond,          # "event" or "no_event"
        "experiment": exp,                  # "baseline" / "fewshot" / "topic_context"
        "context": row["dialog"],           # the dialogue history shown to the model
        "full_prompt": prompt,              # the ENTIRE prompt actually sent, for auditing
        "LLM_generation": generation,       # the model's (cleaned) output
        "next_tutor_utterance": row["next_tutor_utterance"],  # ground truth to compare against
        "next_tutor_utterance_n1": row.get("next_tutor_utterance_n1", None),
        "next_tutor_utterance_n2": row.get("next_tutor_utterance_n2", None),
        "transcriptID": row["transcriptID"],
        "segment_id": row["segment_id"],
        "hyperparameters": str(options),
    })


def resume_state(output_path: str, df: pd.DataFrame):
    """
    If this model's output file already exists (from a previous, possibly
    interrupted run), figure out exactly where to pick back up: which row
    index in `df` to resume from, and what observation_id to continue
    numbering from (so IDs stay unique and don't restart at 0).
    """
    if not os.path.exists(output_path):
        return 0, 0  # nothing to resume, start fresh

    prev = pd.read_csv(output_path)
    prev = prev[prev["experiment"] != "experiment"]  # drop any stray duplicate header rows
    prev["observation_id"] = pd.to_numeric(prev["observation_id"], errors="coerce")
    if len(prev) == 0:
        return 0, 0

    obs_id = int(prev["observation_id"].max()) + 1
    last_transcript = prev["transcriptID"].iloc[-1]
    last_segment = prev["segment_id"].iloc[-1]
    # Find where in `df` that last-completed segment lives, and resume
    # generation from the NEXT row after it.
    matches = df.index[(df["transcriptID"] == last_transcript) & (df["segment_id"] == last_segment)]
    start_index = matches[0] + 1 if len(matches) > 0 else 0
    return start_index, obs_id


def generate_for_model(model_name: str, df: pd.DataFrame, topics_map: dict):
    """
    Run the full generation loop for ONE model, across every segment,
    strategy, condition, and run. Writes results incrementally (every 10
    segments) so progress survives a crash or manual interruption.
    """
    model_tag = model_name.replace(":", "").replace("/", "")
    output_path = f"llm_{model_tag}.csv"
    write_header = not os.path.exists(output_path)

    start_index, obs_id = resume_state(output_path, df)
    print(f"{model_name}: {'resuming from row ' + str(start_index) if start_index else 'starting fresh'} "
          f"(obs_id={obs_id}) -> {output_path}")

    agent = TutorGenerationAgent(model=model_name, temperature=0.2)
    rows = []

    for index, row in tqdm(df.iloc[start_index:].iterrows(),
                            total=len(df) - start_index, desc=model_name):

        # Two independent runs per segment, used later to measure
        # run-to-run consistency (see paper Methods, MAI section).
        for _ in range(2):
            timestamp = datetime.now().isoformat()

            # ── Baseline strategy, both conditions ──────────────────────
            for cond, event_val in [("no_event", None), ("event", row["event_context"])]:
                prompt = make_prompt(row["dialog"], event=event_val)
                try:
                    gen, _, options, _ = agent.chat(prompt, fewshot=False)
                except Exception as e:
                    print(f"Error baseline {cond}: {e}")
                    continue  # skip this one generation, keep going with the rest
                append_row(rows, obs_id, model_name, timestamp, cond, "baseline", prompt, gen, options, row)
                obs_id += 1

            # ── Few-shot strategy, both conditions ──────────────────────
            for cond, event_val in [("no_event", None), ("event", row["event_context"])]:
                prompt = make_prompt_fewshot(row["dialog"], event=event_val)
                try:
                    gen, _, options, _ = agent.chat(prompt, fewshot=True)
                except Exception as e:
                    print(f"Error fewshot {cond}: {e}")
                    continue
                append_row(rows, obs_id, model_name, timestamp, cond, "fewshot", prompt, gen, options, row)
                obs_id += 1

            # ── Topic-context strategy, both conditions ─────────────────
            # Only generated if this session actually has extracted topics
            # (some sessions might have failed topic extraction upstream).
            math_topics = topics_map.get(row["transcriptID"], "")
            if isinstance(math_topics, str) and math_topics.strip() != "":
                for cond, event_val in [("no_event", None), ("event", row["event_context"])]:
                    prompt = make_prompt_topic_context(row["dialog"], math_topics, event=event_val)
                    try:
                        gen, _, options, _ = agent.chat(prompt, fewshot=False)
                    except Exception as e:
                        print(f"Error topic_context {cond}: {e}")
                        continue
                    append_row(rows, obs_id, model_name, timestamp, cond, "topic_context", prompt, gen, options, row)
                    obs_id += 1

        # Checkpoint: write to disk every 10 segments instead of holding
        # everything in memory until the very end (safer for long runs).
        if (index - start_index + 1) % 10 == 0 and rows:
            pd.DataFrame(rows).to_csv(output_path, mode="a", header=write_header, index=False)
            write_header = False
            rows.clear()

    # Flush whatever's left over after the loop finishes.
    if rows:
        pd.DataFrame(rows).to_csv(output_path, mode="a", header=write_header, index=False)

    print(f"{model_name}: done -> {output_path}")


def main():
    args = parse_args()
    models = [m.strip() for m in args.models.split(",")]

    print(f"Loading: {args.input}")
    df = pd.read_csv(args.input)
    print(f"  {len(df)} rows | Models: {models}")

    if "next_tutor_utterance_n1" not in df.columns:
        print("  WARNING: n+1/n+2 columns missing.")

    # Build a lookup: transcriptID -> that session's math topics string,
    # so we don't need to re-read/re-filter the topics file per segment.
    topics_df = pd.read_csv(args.topics)
    topics_map = dict(zip(topics_df["transcriptID"], topics_df["math_topics"]))

    for model_name in models:
        generate_for_model(model_name, df, topics_map)

    print("\nDone.")


if __name__ == "__main__":
    main()